In [ ]:
!pip install faiss-cpu
!pip install umap-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 2.8 MB/s eta 0:00:00


In [ ]:
pip install --upgrade numba


In [ ]:
!pip uninstall -y umap-learn numba
!pip install --no-cache-dir umap-learn numba


Found existing installation: umap-learn 0.5.7
Uninstalling umap-learn-0.5.7:
  Successfully uninstalled umap-learn-0.5.7
Found existing installation: numba 0.61.0
Uninstalling numba-0.61.0:
  Successfully uninstalled numba-0.61.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 53.5 MB/s eta 0:00:00


In [ ]:
import faiss
import json
import numpy as np
import umap
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

# Load FAISS index
faiss_index = faiss.read_index("/content/faiss_index_e5_V2.idx")
print(f"Index loaded with {faiss_index.ntotal} vectors")

# Load metadata
with open("/content/metadata_store_e5_V2.json", "r") as f:
    metadata_store = json.load(f)
print(f"Metadata loaded with {len(metadata_store)} entries")

# 🔹 **Enable FAISS reconstruction**
faiss_index.make_direct_map()  # Enables reconstruct()
num_vectors = faiss_index.ntotal
embedding_dim = faiss_index.d

# Extract stored embeddings
stored_vectors = np.zeros((num_vectors, embedding_dim), dtype=np.float32)
for i in range(num_vectors):
    stored_vectors[i] = faiss_index.reconstruct(i)

# Perform UMAP dimensionality reduction
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embeddings_2d = umap_model.fit_transform(stored_vectors)

# Convert metadata to DataFrame
df = pd.DataFrame(metadata_store)

# Rename columns
df = df.rename(columns={'project_type': 'Bank', 'sector': 'Sector', 'data_id': 'Doc_ID'})

# Add UMAP coordinates to DataFrame
df['UMAP1'] = embeddings_2d[:, 0]
df['UMAP2'] = embeddings_2d[:, 1]

print("Shape of embeddings:", stored_vectors.shape)
print("Shape of UMAP embeddings:", embeddings_2d.shape)
print("Number of rows in DataFrame:", len(df))

# Verify data
print("\nFirst few rows of DataFrame with UMAP coordinates:")
print(df[['Bank', 'Sector', 'UMAP1', 'UMAP2']].head())

# 🔹 **Perform KMeans clustering for NMI scoring**
num_clusters = df['Sector'].nunique()  # Match number of unique sectors
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(embeddings_2d)

# 🔹 **Calculate NMI Score**
true_labels = df['Sector'].astype("category").cat.codes  # Convert sectors to numerical labels
predicted_labels = df['Cluster']
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)

print(f"\n✅ NMI Score for Clustering: {nmi_score:.4f}")

# Define Colors and Shapes
BANK_SHAPES = {
    'World Bank': 'circle',
    'AFDB': 'diamond',
    'AIIB': 'square'
}

SECTOR_COLORS = {
    'ENERGY AND EXTRACTIVES': 'rgba(255, 99, 132, 0.7)',
    'TRANSPORTATION': 'rgba(54, 162, 235, 0.7)',
    'WATER, SANITATION AND WASTE MANAGEMENT': 'rgba(85, 192, 192, 0.7)',
    'FINANCIAL SECTOR': 'rgba(153, 102, 255, 0.7)',
    'Finance': 'rgba(255, 159, 64, 0.7)',
    'HEALTH': 'rgba(255, 205, 86, 0.7)',
    'MULTI-SECTOR': 'rgba(201, 203, 207, 0.7)',
    'AGRICULTURE, FISHING AND FORESTRY': 'rgba(102, 205, 170, 0.7)',
    'INFORMATION AND COMMUNICATIONS TECHNOLOGIES': 'rgba(255, 99, 71, 0.7)',
    'SOCIAL SUPPORT': 'rgba(50, 150, 235, 0.7)',
    'INDUSTRY, TRADE AND SERVICES': 'rgba(255, 179, 64, 0.7)',
    'PUBLIC ADMINISTRATION': 'rgba(85, 192, 192, 0.7)',
    'EDUCATION': 'rgba(169, 169, 169, 0.7)',
    'Other': 'rgba(255, 255, 255, 0.7)'
}

# Create interactive Plotly scatter plot
fig = go.Figure()
traces_info = []

for bank in df['Bank'].unique():
    for sector in df['Sector'].unique():
        mask = (df['Bank'] == bank) & (df['Sector'] == sector)
        if any(mask):
            fig.add_trace(
                go.Scatter(
                    x=df[mask]['UMAP1'],
                    y=df[mask]['UMAP2'],
                    mode='markers',
                    name=f"{bank} - {sector}",
                    marker=dict(
                        size=6,
                        symbol=BANK_SHAPES.get(bank, 'circle'),
                        color=SECTOR_COLORS.get(sector, 'rgba(169, 169, 169, 0.7)'),
                        opacity=0.7,
                        line=dict(width=1, color='rgba(0, 0, 0, 0.5)')
                    ),
                    hovertemplate="<b>Bank:</b> %{customdata[0]}<br>"
                                  "<b>Sector:</b> %{customdata[1]}<br>"
                                  "<b>Document ID:</b> %{customdata[2]}<br>"
                                  "<extra></extra>",
                    customdata=df[mask][['Bank', 'Sector', 'Doc_ID']].values
                )
            )
            traces_info.append({'bank': bank, 'sector': sector})

# Create Bank selection buttons
bank_buttons = []
for bank in sorted(df['Bank'].unique()):
    visibility = [trace['bank'] == bank for trace in traces_info]
    bank_buttons.append(
        dict(
            label=bank,
            method="update",
            args=[{"visible": visibility}]
        )
    )

# Add "Show All" button for Bank selection
bank_buttons.insert(0, dict(
    label="All Banks",
    method="update",
    args=[{"visible": [True] * len(traces_info)}]
))

# Create Sector selection buttons
sector_buttons = []
for sector in sorted(df['Sector'].unique()):
    visibility = [trace['sector'] == sector for trace in traces_info]
    sector_buttons.append(
        dict(
            label=sector,
            method="update",
            args=[{"visible": visibility}]
        )
    )

# Add "Show All" button for Sector selection
sector_buttons.insert(0, dict(
    label="All Sectors",
    method="update",
    args=[{"visible": [True] * len(traces_info)}]
))

# Update layout with dropdown menus
fig.update_layout(
    updatemenus=[
        dict(
            buttons=bank_buttons,
            direction="down",
            showactive=True,
            x=0.1,
            y=1.1,
            xanchor="left",
            yanchor="top",
            name="Bank"
        ),
        dict(
            buttons=sector_buttons,
            direction="down",
            showactive=True,
            x=0.4,
            y=1.1,
            xanchor="left",
            yanchor="top",
            name="Sector"
        )
    ],
    title={
        'text': "Development Bank Embeddings (UMAP Projection)<br><sub>Shapes: World Bank (○) | AFDB (◇) | AIIB (□)</sub>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    margin=dict(t=150),
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.9)"
    )
)

# Show the interactive plot
fig.show()


Index loaded with 30633 vectors
Metadata loaded with 30633 entries


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape of embeddings: (30633, 1024)
Shape of UMAP embeddings: (30633, 2)
Number of rows in DataFrame: 30633

First few rows of DataFrame with UMAP coordinates:
   Bank Sector      UMAP1     UMAP2
0  AFDB  Other  14.425536  7.347126
1  AFDB  Other  14.444284  7.392290
2  AFDB  Other  14.385970  7.376587
3  AFDB  Other  14.437625  7.426958
4  AFDB  Other  14.449193  7.431521

✅ NMI Score for Clustering: 0.3956
